# 1. The World Bank's international debt data

Countries borrow for many of the same reasons people do, but at a much larger scale. Governments take on debt to fund infrastructure, stabilize their economies, and invest in long-term growth. [The World Bank](https://www.worldbank.org) is one of the largest lenders to developing countries.

This project analyzes World Bank data on the debt (in USD) owed by developing countries across several debt categories. The goal is to answer questions like:

- What is the total debt owed by all countries in the dataset?
- Which country owes the most, and what does that debt consist of? 
- What is the average debt across different debt indicators?

To start, I connect to a local DuckDB containing the 'international_debt' table and preview the first 10 rows to get a sense of its structure.

In [10]:
%pip install duckdb duckdb-engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 31.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
%load_ext sql

In [2]:
%sql duckdb:///debt.duckdb

Connecting to 'duckdb:///debt.duckdb'

In [3]:
%%sql
CREATE TABLE international_debt AS
SELECT * FROM read_csv_auto('international_debt.csv');

Running query in 'duckdb:///debt.duckdb'

Count


In [8]:
%%sql
SELECT * 
FROM international_debt 

Running query in 'duckdb:///debt.duckdb'

country_name,country_code,indicator_name,indicator_code,debt
Afghanistan,AFG,"Disbursements on external debt, long-term (DIS, current US$)",DT.DIS.DLXF.CD,72894453.7
Afghanistan,AFG,"Interest payments on external debt, long-term (INT, current US$)",DT.INT.DLXF.CD,53239440.1
Afghanistan,AFG,"PPG, bilateral (AMT, current US$)",DT.AMT.BLAT.CD,61739336.9
Afghanistan,AFG,"PPG, bilateral (DIS, current US$)",DT.DIS.BLAT.CD,49114729.4
Afghanistan,AFG,"PPG, bilateral (INT, current US$)",DT.INT.BLAT.CD,39903620.1
Afghanistan,AFG,"PPG, multilateral (AMT, current US$)",DT.AMT.MLAT.CD,39107845.0
Afghanistan,AFG,"PPG, multilateral (DIS, current US$)",DT.DIS.MLAT.CD,23779724.3
Afghanistan,AFG,"PPG, multilateral (INT, current US$)",DT.INT.MLAT.CD,13335820.0
Afghanistan,AFG,"PPG, official creditors (AMT, current US$)",DT.AMT.OFFT.CD,100847181.9
Afghanistan,AFG,"PPG, official creditors (DIS, current US$)",DT.DIS.OFFT.CD,72894453.7


# 2. Finding The Number of Distinct Countries

The first 10 rows show Afghanistan's debt across several indicators. Country names repeat because most countries carry debt in more than one category, so the row count alone doesn't tell us how many countries the dataset covers.

Knowing that number gives context for everything that follows, such as totals and average per country. Here, I count the distinct countries in the table.

In [6]:
%%sql
SELECT
    COUNT(DISTINCT country_name) AS total_distinct_countries
FROM international_debt

Running query in 'duckdb:///debt.duckdb'

total_distinct_countries
124


# 3. Finding the Distinct Debt Indicators

The dataset covers 124 countries. Each row also has an `indicator_name`, which describes what the debt is for, and an `indicator_code`, a short ID for that category. Listing the distinct indicators shows the different types of debt a country can carry, which will be useful when comparing countries later.

In [9]:
%%sql 
SELECT DISTINCT indicator_code AS distinct_debt_indicators
FROM international_debt
ORDER BY distinct_debt_indicators

Running query in 'duckdb:///debt.duckdb'

distinct_debt_indicators
DT.AMT.BLAT.CD
DT.AMT.DLXF.CD
DT.AMT.DPNG.CD
DT.AMT.MLAT.CD
DT.AMT.OFFT.CD
DT.AMT.PBND.CD
DT.AMT.PCBK.CD
DT.AMT.PROP.CD
DT.AMT.PRVT.CD
DT.DIS.BLAT.CD


# 4. Total Debt Owed by All Countries

With the debt categories mapped out, the next step is the big picture: how much debt, in USD, do all the countries in the dataset owe combined? This gives a sense of the overall debt burden across developing economies.

In [10]:
%%sql 
SELECT
    ROUND(SUM(debt) / 1000000, 2) AS total_debt
FROM international_debt; 

Running query in 'duckdb:///debt.duckdb'

total_debt
3079734.49


# 5. Country with the Highest Total Debt

The combined debt in the dataset is over 3 trillion USD, a number that is hard to grasp intuitively. To make it more concrete, let's look at individual countries.

Which country owes the most? Each country's debt is spread across several indicators, so i'll sum those up per country and rank the results. The country at the top says something about where debt is most concentrated. Breaking its debt down by category would be a natural follow-up, but i'll save that for later

In [11]:
%%sql 
SELECT
    country_name,
    SUM(debt) AS total_debt
FROM international_debt
GROUP BY country_name
ORDER BY total_debt DESC
LIMIT 1;

Running query in 'duckdb:///debt.duckdb'

country_name,total_debt
China,285793494734.2


# 6. Avg. Debt Across Indicators 

The country with the highest total debt is China. For a more detailed breakdown of its debt, see the [World Bank's China profile](https://datatopics.worldbank.org/debt/ids/country/CHN).

Now that we have a high-level view of its dataset, we can dig into the distribution. For each debt indicator, what is the average amount owed? Comparing averages across indicators shows which types of debt are typically largest, and which ones account for most of the borrowing.

In [14]:
%%sql
SELECT
    indicator_code AS debt_indicator,
    indicator_name,
    AVG(debt) AS average_debt
FROM international_debt
GROUP BY debt_indicator, indicator_name
ORDER BY average_debt DESC
LIMIT 10;

Running query in 'duckdb:///debt.duckdb'

debt_indicator,indicator_name,average_debt
DT.AMT.DLXF.CD,"Principal repayments on external debt, long-term (AMT, current US$)",5904868401.499195
DT.AMT.DPNG.CD,"Principal repayments on external debt, private nonguaranteed (PNG) (AMT, current US$)",5161194333.812659
DT.DIS.DLXF.CD,"Disbursements on external debt, long-term (DIS, current US$)",2152041216.8902435
DT.DIS.OFFT.CD,"PPG, official creditors (DIS, current US$)",1958983452.8598351
DT.AMT.PRVT.CD,"PPG, private creditors (AMT, current US$)",1803694101.963265
DT.INT.DLXF.CD,"Interest payments on external debt, long-term (INT, current US$)",1644024067.6508067
DT.DIS.BLAT.CD,"PPG, bilateral (DIS, current US$)",1223139290.39823
DT.INT.DPNG.CD,"Interest payments on external debt, private nonguaranteed (PNG) (INT, current US$)",1220410844.4215188
DT.AMT.OFFT.CD,"PPG, official creditors (AMT, current US$)",1191187963.083064
DT.AMT.PBND.CD,"PPG, bonds (AMT, current US$)",1082623947.6536236


# 7. Highest Principle Repayments on Long-Term Debt

The indicator `DT.AMT.DLXF.CD` has the highest average, and it covers principle repayments on long-term external debt. Countries take on long-term debt to raise capital upfront, then repay it over many years. More detail is in the [World Bank data catalog](https://datacatalog.worldbank.org/principal-repayments-external-debt-long-term-amt-current-us-0).

The averages also show a large gap after the first two indicators. That suggest these two categories account for most of what countries owe, though the averages alone don't prove it.

Next, I look at which country has the highest `DT.AMT.DLXF.CD` debt. Since countries face different economic conditions, this shows where long-term repayment obligations are more concentrated.

In [16]:
%%sql
SELECT 
    country_name, 
    indicator_name
FROM international_debt
WHERE debt = (SELECT 
                 MAX(debt)
             FROM international_debt
             WHERE indicator_code = 'DT.AMT.DLXF.CD');

Running query in 'duckdb:///debt.duckdb'

country_name,indicator_name
China,"Principal repayments on external debt, long-term (AMT, current US$)"


# 8. The Most Common Debt Indicator

China has the highest `DT.AMT.DLXF.CD` debt, which matches the [World Banks's own data](https://data.worldbank.org/indicator/DT.AMT.DLXF.CD?end=2018&most_recent_value_desc=true). Checking results against an outside source like this is good practice, since it confirms the analysis is on track

Long-term debt has the highest average, but is it also the indicator that shows up most often across countries? Counting how many countries have a row for each indicator answers that.

In [17]:
%%sql
SELECT indicator_code, COUNT(indicator_code) AS indicator_count
FROM international_debt
GROUP BY indicator_code
ORDER BY indicator_count DESC, indicator_code DESC
LIMIT 20;

Running query in 'duckdb:///debt.duckdb'

indicator_code,indicator_count
DT.INT.OFFT.CD,124
DT.INT.MLAT.CD,124
DT.INT.DLXF.CD,124
DT.AMT.OFFT.CD,124
DT.AMT.MLAT.CD,124
DT.AMT.DLXF.CD,124
DT.DIS.DLXF.CD,123
DT.INT.BLAT.CD,122
DT.DIS.OFFT.CD,122
DT.AMT.BLAT.CD,122


# 9. Other Debt Indicators & Conclusion

Six debt indicators appear for all 124 countries in the dataset, including `DT.AMT.DLXF.CD`. This shows that every country carries long-term debt, though it doesn't by itself point to a shared economic problem. Borrowing is normal, and the data doesn't say whether any of it is a burden.

To look at the amounts again, I find the largest single debt each country holds. This shows which type of debt dominates for each country and can point to differences in how they borrow.

### Conclusion

This project analyzed World Bank data on debt owed by developing countries. Using SQL, I:

- counted the countries and debt indicators in the dataset,
- calculated the total debt owed across all countries,
- identified the country with the highest total debt (China),
- compared average debt across indicators, and
- found which indicators are most common across countries.

Long-term principle repayments were the largest category, and China had the highest debt in it, which matched the World Bank's published figures.

In [18]:
%%sql 
SELECT country_name,
MAX(debt) AS maximum_debt
FROM international_debt
GROUP BY country_name
ORDER BY maximum_debt DESC
LIMIT 10

Running query in 'duckdb:///debt.duckdb'

country_name,maximum_debt
China,96218620835.7
Brazil,90041840304.1
Russian Federation,66589761833.5
Turkey,51555031005.8
South Asia,48756295898.2
Least developed countries: UN classification,40160766261.6
IDA only,34531188113.2
India,31923507000.8
Indonesia,30916112653.8
Kazakhstan,27482093686.4
